# J5-03 · Conformité appliquée : AI Act, RGPD, NIS2 dans le code

**Formation « Orchestration d'agents & Monitoring : LangGraph + Langfuse (stack 100 % locale) »**
Jour 5 · créneau 12h45–13h45 · ⏱️ 1h (dont 30 min de pratique)

| | |
|---|---|
| 🎯 **Objectifs** | 1. Traduire trois textes en **exigences techniques** : informer (AI Act art. 50), minimiser et conserver (RGPD), tracer et notifier (NIS2) · 2. Situer HelpDesk-Ops dans la **classification de risque** de l'AI Act · 3. Produire les preuves : information affichée, réponses marquées, purge planifiée, chronologie d'incident |
| 🧱 **Fil rouge** | Le jalon J5 et les traces de la red-team du matin |
| ✅ **Prérequis** | J5-01, J5-02 ; J2-01 (checkpoints), J4-02 (purge des traces) |

> ⚠️ **Ce notebook n'est pas un avis juridique.** Il montre comment un développeur rend une exigence **vérifiable**. Les dates et références ci-dessous ont été vérifiées le **22/10/2026** ; plusieurs l'ont été sur des sources secondaires et portent la mention **à vérifier** : relisez le texte officiel avant de les citer à un client.

## 1. Mise en route

### [Code] Le socle du fil rouge : `src/helpdesk_ops`

Le code écrit et expliqué en J1-01 (lecture de `.env`, masquage des données personnelles, client Langfuse, création du modèle) est rangé dans le paquet **`helpdesk_ops`**, installé par `uv sync`. Chaque notebook reste **autonome** : la cellule suivante suffit.

> 💡 **Le « Tips » du Formateur :** le paquet est installé en mode *éditable* : une modification dans `src/helpdesk_ops` est prise en compte après *Kernel → Restart*. Le client Langfuse est un singleton par clé publique : en cas de doute, redémarrez le kernel.

In [ ]:
import os

from langfuse import observe, propagate_attributes

from helpdesk_ops.config import creer_llm
from helpdesk_ops.observabilite import init_observabilite
from helpdesk_ops.texte import sans_accents

langfuse, langfuse_handler = init_observabilite()   # .env chargé, masquage PII actif, clés vérifiées
llm = creer_llm("agent")                             # MODEL_AGENT, temperature=0, num_ctx=16384
print(f"✅ Langfuse : {os.environ['LANGFUSE_BASE_URL']} | modèle : {llm.model}")

## 2. Trois textes, trois questions

### [Théorie] Le code de la route, pas le manuel du moteur

Ces textes ne disent pas **comment** coder ; ils disent ce qui doit être **vrai** et **prouvable** : l'utilisateur sait qu'il parle à une IA, les données ne traînent pas, un incident se reconstitue. Le développeur choisit le moyen et garde la preuve.

| Texte | Question pour HelpDesk-Ops | Réponse dans le code |
|---|---|---|
| **AI Act** (Règl. (UE) 2024/1689), art. 50 | L'utilisateur sait-il qu'il parle à une IA ? Une réponse générée est-elle identifiable ? | `MESSAGE_IA` affiché, réponses **marquées** (`conformite.marquer`) |
| **RGPD** (Règl. (UE) 2016/679), art. 5, 17, 25 | Que garde-t-on, combien de temps, peut-on l'effacer ? | `PIIMiddleware`, `mask=`, `purge_planifiee`, effacement par session |
| **NIS2** (dir. (UE) 2022/2555), en Belgique loi du 26/04/2024 | Peut-on dire ce qui s'est passé, quand, et le notifier dans les délais ? | traces Langfuse, journal d'audit, `chronologie_incident` |

### [Théorie] Les faits, avec leur statut (vérifiés le 22/10/2026)

| Fait | Statut |
|---|---|
| AI Act art. 50 (transparence) applicable depuis le **02/08/2026** | confirmé (sources secondaires concordantes) ; texte EUR-Lex à relire |
| 50(1) : les systèmes destinés à interagir avec des personnes sont conçus pour qu'elles **sachent** qu'elles parlent à une IA, sauf si c'est évident ; obligation du **fournisseur** du système | à vérifier (texte non relu dans la session de vérification) |
| 50(2) : **marquage machine-lisible** des contenus générés (texte compris) ; **jusqu'au 02/12/2026** pour les systèmes mis sur le marché **avant** le 02/08/2026 | délai confirmé (sources secondaires) ; article exact de l'Omnibus à citer depuis EUR-Lex |
| « Omnibus numérique » IA : **Règl. (UE) 2026/1744**, JO du 24/07/2026, en vigueur le 27/07/2026 | confirmé (sources secondaires concordantes) |
| Obligations « haut risque » reportées : **annexe III au 02/12/2027**, **annexe I au 02/08/2028** | confirmé (sources secondaires) |
| Art. 4 (culture IA) : l'obligation directe des fournisseurs et déployeurs est remplacée par un devoir de soutien de la Commission et des États membres | à vérifier (une source secondaire) |
| Un rectificatif (version allemande) de l'Omnibus est en cours | à vérifier au JO, date non confirmée |
| NIS2 Belgique : loi du 26/04/2024, autorité : **CCB** ; référentiels **CyFun** ou **ISO/IEC 27001** ; entités essentielles : preuves de démarche au **18/04/2026**, certification ou niveau requis au **18/04/2027** | à vérifier sur ccb.belgium.be (site inaccessible lors de la vérification) |
| Notification d'incident significatif : alerte précoce **24 h**, notification **72 h**, rapport final **1 mois** (dir. 2022/2555, art. 23) | à vérifier (rappel, texte non relu) |

## 3. AI Act : où se situe HelpDesk-Ops ?

### [Théorie] Une classification par usage, pas par technologie

| Niveau | Exemple | HelpDesk-Ops ? |
|---|---|---|
| Pratiques interdites (art. 5) | notation sociale, manipulation | non |
| **Haut risque** (annexes I et III) | recrutement, **évaluation des travailleurs**, éducation (accès, évaluation des apprenants), crédit… | **non par nature** : il aide à résoudre des incidents informatiques |
| **Transparence** (art. 50) | systèmes qui interagissent avec des personnes, contenus générés | **oui** : c'est notre cas |
| Risque minimal | filtre anti-spam | — |

⚠️ **L'usage peut faire basculer.** Si l'on se servait des traces de HelpDesk-Ops pour **évaluer ou surveiller les collaborateurs** (« qui pose trop de questions ? »), on entrerait dans l'annexe III (emploi, point 4). Même logique pour une formation : **noter des apprenants avec un LLM** relève de l'annexe III (éducation, point 3, à vérifier). C'est pourquoi le quiz de cette formation sert au formateur, pas à noter, et que le QCM final est corrigé par une personne.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

from helpdesk_ops.conformite import MESSAGE_IA, est_marque
from helpdesk_ops.jalons.j5 import arepondre_j5, construire_helpdesk_j5

helpdesk = construire_helpdesk_j5(llm, checkpointer=InMemorySaver(), modele=llm.model)

# Une demande hors périmètre : le chemin le plus court (garde-fou seulement), idéal pour regarder la forme
with propagate_attributes(session_id="j5-03-art50", user_id="collab-demo", tags=["jour-5", "ai-act"]):
    reponse = await arepondre_j5(helpdesk, "Écris-moi un poème sur les serveurs", thread_id="j5-03-art50",
                                 user_id="collab-demo", callbacks=[langfuse_handler])
print(reponse.content, "\n")
print("marquée :", est_marque(reponse))
print(reponse.response_metadata["ia_act_art50"])

**👀 Ce que vous devez observer :** la première réponse de la conversation **commence par l'information** « Vous échangez avec HelpDesk-Ops, un assistant d'intelligence artificielle… » (art. 50(1)), puis le refus du garde-fou. Et la réponse porte, dans ses métadonnées, un **marquage** : `genere_par_ia`, système, version, modèle, horodatage. Ces métadonnées suivent le message dans l'API et dans Streamlit (J5-04), où l'information reste aussi affichée en permanence.

> 💡 **Le « Tips » du Formateur :** l'art. 50(2) vise surtout les contenus de synthèse diffusés (images, textes publiés) et renvoie à des formats techniques que la Commission précise par des lignes directrices et des codes de pratique. Pour un help-desk interne, le marquage en métadonnées est une **bonne pratique** peu coûteuse ; le format exact est à aligner sur ces documents, **à vérifier** avant de l'affirmer conforme.

## 4. RGPD : le registre, puis la purge

### [Théorie] La fiche du traitement « HelpDesk-Ops » (proposition)

| Rubrique du registre (art. 30) | HelpDesk-Ops |
|---|---|
| Finalité | traiter les demandes d'assistance informatique des collaborateurs |
| Base légale | à déterminer avec le DPO (intérêt légitime ou exécution du contrat de travail) |
| Personnes concernées | collaborateurs de NordTech |
| Données | pseudonyme, contenu des demandes (peut contenir des données personnelles malgré les filtres), traces techniques |
| Destinataires | équipe help-desk N1/N2 ; aucun tiers : modèles et traces **hébergés en local** |
| Durées | conversations (checkpoints) **30 jours** ; traces **30 jours** ; journal d'audit des outils : selon la politique de sécurité |
| Mesures | minimisation (pseudonyme, `PIIMiddleware`, `mask=`), accès restreint à Langfuse, purge planifiée, effacement sur demande, chiffrement du disque |

Les durées sont celles de `helpdesk_ops.conformite.CONSERVATION` : **une seule source de vérité**, utilisée par le code et recopiée au registre.

In [ ]:
from langgraph.checkpoint.sqlite import SqliteSaver

from helpdesk_ops.conformite import CONSERVATION, purge_planifiee
from helpdesk_ops.persistance import chemin_checkpoints

print("conservation :", {k: f"{v.days} j" for k, v in CONSERVATION.items()})
with SqliteSaver.from_conn_string(str(chemin_checkpoints())) as saver:   # les conversations des jours 2 à 5
    a_purger = purge_planifiee(langfuse, saver, simulation=True)
print({k: len(v) for k, v in a_purger.items()}, "élément(s) à purger aujourd'hui (simulation)")

**👀 Ce que vous devez observer :** probablement **0** partout : les données du cours ont moins de 30 jours. Le résultat compte moins que le geste : une **simulation** journalisée avant la première vraie purge, puis la même fonction planifiée chaque nuit avec `simulation=False` (cron, tâche planifiée, ou un job de CI planifié).

**L'effacement sur demande** (art. 17) est déjà en place : `purger_traces(session_id=…)` (J4-02), `saver.delete_thread(thread_id)` (J2-01), et le bouton « Effacer cette conversation » de l'interface Streamlit (J5-04) qui fait les deux.

## 5. NIS2 : reconstituer un incident à partir des traces

Depuis ce matin, la stack a enregistré des erreurs et des alertes : le modèle « en panne » de J5-01, les essais de l'exercice 5.1, et celles du lab red-team s'il a été tracé dans Langfuse. Supposons que l'équipe sécurité demande : « que s'est-il passé aujourd'hui ? ».

In [ ]:
from datetime import datetime, timezone

import pandas as pd

from helpdesk_ops.conformite import chronologie_incident

aujourd_hui = datetime.now(timezone.utc).replace(hour=0, minute=0, second=0, microsecond=0)
evenements = pd.DataFrame(chronologie_incident(langfuse, debut=aujourd_hui))
if evenements.empty:
    print("aucune alerte aujourd'hui : lancez d'abord J5-02")
else:
    evenements["debut"] = pd.to_datetime(evenements["debut"]).dt.strftime("%H:%M:%S")
    print(evenements.groupby(["niveau", "nom"]).size().rename("nombre").to_string(), "\n")
    print(evenements[["debut", "niveau", "nom", "message"]].head(15).to_string(index=False))

**👀 Ce que vous devez observer :** une chronologie **horodatée** : quels nœuds, modèles ou outils ont alerté, combien de fois, avec quel message. En préparation, on y lisait par exemple les deux `ERROR` « model 'modele-inexistant:latest' not found (status code: 404) » de J5-01, à la seconde près. C'est la matière première d'une notification ; le jugement « incident significatif ou non » et le rapport restent humains.

### [Théorie] De la trace à la notification

| Étape (délais à vérifier, § 2) | Ce que les traces apportent | Ce qu'elles n'apportent pas |
|---|---|---|
| Alerte précoce (24 h) | heure de début, composants touchés, nombre d'occurrences | l'impact métier, l'intention de l'attaquant |
| Notification (72 h) | chronologie, sessions et versions concernées (prompt, modèle, code) | l'évaluation de gravité |
| Rapport final (1 mois) | preuves de la correction : mêmes attaques, taux après correctif (J5-02), gate de CI (J5-04) | la décision de clôture |

> 💡 **Le « Tips » du Formateur :** pour que cette chronologie soit recevable, trois conditions : les horloges synchronisées, des traces **non modifiables** par l'équipe qui les produit (droits Langfuse), et une **durée de conservation** compatible avec les délais d'enquête. Ce dernier point peut contredire les 30 jours du RGPD : c'est un arbitrage à documenter avec le DPO et le RSSI, pas à trancher dans le code.

## 6. La check-list conformité du projet final

| ✔ | Exigence | Preuve attendue |
|---|---|---|
| ☐ | Information « IA » visible au premier échange (art. 50(1)) | capture de l'interface ou du premier message |
| ☐ | Réponses marquées (métadonnées) | `est_marque(reponse)` |
| ☐ | Pas de donnée personnelle inutile vers le modèle ni vers les traces | `PIIMiddleware` + `mask=` ; une trace ouverte en démonstration |
| ☐ | Durée de conservation codée et purge testée en simulation | `purge_planifiee(..., simulation=True)` |
| ☐ | Effacement d'une conversation | traces et checkpoint supprimés |
| ☐ | Une alerte de sécurité visible dans Langfuse | observation `WARNING` ou `ERROR` filtrée |
| ☐ | Aucun usage d'évaluation de personnes | le dire explicitement dans la démo |

## 🧭 Récap

| Exigence | Dans HelpDesk-Ops |
|---|---|
| AI Act art. 50(1) : informer | `MESSAGE_IA` au 1ᵉʳ échange + bandeau permanent dans l'interface |
| AI Act art. 50(2) : marquer | `conformite.marquer` → `response_metadata["ia_act_art50"]` |
| RGPD art. 5(1)(c) : minimiser | pseudonyme, `PIIMiddleware`, `mask=` |
| RGPD art. 5(1)(e) : conserver peu | `CONSERVATION`, `purge_planifiee` |
| RGPD art. 17 : effacer | `purger_traces(session_id=…)` + `delete_thread` |
| NIS2 : tracer, notifier | traces, journal d'audit, `chronologie_incident` |

### ✅ Quiz express

1. HelpDesk-Ops est-il un système à haut risque au sens de l'AI Act ? Qu'est-ce qui pourrait le faire basculer ?
2. Où l'information « vous parlez à une IA » doit-elle apparaître, et quand ?
3. Pourquoi la durée de conservation est-elle une constante du code ET une ligne du registre ?
4. Que manque-t-il aux traces Langfuse pour rédiger seul une notification NIS2 ?

<details>
<summary>Réponses</summary>

1. Non par nature (assistance informatique). Il basculerait s'il servait à évaluer ou surveiller des personnes (annexe III, emploi ; éducation pour des apprenants).
2. Dès le premier échange, de façon claire (art. 50(1)) ; dans l'interface, le bandeau reste visible pendant toute la conversation.
3. Le registre dit ce qui est promis ; le code le fait. Une seule valeur, recopiée, évite qu'ils divergent.
4. L'impact métier, la gravité, l'intention, la décision : les traces donnent le quoi et le quand, pas le jugement.
</details>